# Data Ingestion, Summarization and Storage

This notebook walks through the full data pipeline:

1. **Fetch & chunk** — pull Wikipedia articles and split into ~250-word paragraphs
2. **Summarize** — generate three style variants per chunk via OpenRouter (shorten / professional / informal)
3. **Inspect** — quick sanity checks on the output

The output JSONL (`data/summaries_v4.jsonl`) feeds the training pipeline in `02_lora_laplace_training.ipynb`.

All heavy-lifting is done by `src.data_pipeline` — this notebook only sets parameters and calls that module.

## 1. Configuration

In [1]:
import os, sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR       = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
TITLES_FILE    = DATA_DIR / "wikipedia.lst"
CHUNKS_FILE    = DATA_DIR / "wikipedia_chunks.jsonl"
SUMMARIES_FILE = DATA_DIR / "summaries_v4.jsonl"

# ── Chunking ──────────────────────────────────────────────────────────────────
APPROX_WORDS = 250

# ── OpenRouter ────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY  = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL    = "openai/gpt-4o-mini"
SUMMARY_TEMPERATURE = 0.7
SUMMARY_MAX_TOKENS  = 200
WORKERS             = 4
N_MAX               = None   # set to an int to limit chunks processed

assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY before running this notebook."
print("Config OK")

Config OK


## 2. Define Wikipedia titles

In [2]:
# Titles are read from TITLES_FILE (data/wikipedia.lst).
# Lines starting with # are comments and will be skipped.
titles = [
    t.strip()
    for t in TITLES_FILE.read_text().splitlines()
    if t.strip() and not t.strip().startswith("#")
]
print(f"Loaded {len(titles)} titles from {TITLES_FILE}")

Loaded 77 titles from /Users/disipio/development/summarizer-uncertainty-ml/data/wikipedia.lst


## 3. Fetch and chunk Wikipedia articles

In [3]:
from tqdm.auto import tqdm
from src.data_pipeline import build_session, fetch_and_chunk_titles, write_jsonl
from src.nltk_setup import ensure_sentence_tokenizer

ensure_sentence_tokenizer(download=True)

titles = [
    t.strip()
    for t in TITLES_FILE.read_text().splitlines()
    if t.strip() and not t.strip().startswith("#")
]
session = build_session()
written = 0

CHUNKS_FILE.unlink(missing_ok=True)
for chunk in tqdm(fetch_and_chunk_titles(titles, approx_words=APPROX_WORDS, session=session),
                  desc="Fetching & chunking"):
    write_jsonl(str(CHUNKS_FILE), chunk)
    written += 1

print(f"Wrote {written} chunks to {CHUNKS_FILE}")

Fetching & chunking: 0it [00:00, ?it/s]

Wrote 2999 chunks to /Users/disipio/development/summarizer-uncertainty-ml/data/wikipedia_chunks.jsonl


## 4. Generate summaries via OpenRouter

In [4]:
import json
from src.data_pipeline import read_jsonl, summarize_chunks, SUMMARY_STYLES

chunks = list(read_jsonl(str(CHUNKS_FILE)))
if N_MAX is not None:
    chunks = chunks[:N_MAX]
print(f"Summarizing {len(chunks)} chunks × {len(SUMMARY_STYLES)} styles = {len(chunks)*len(SUMMARY_STYLES)} API calls")

SUMMARIES_FILE.unlink(missing_ok=True)
written = errors = 0

for result in tqdm(
    summarize_chunks(chunks, model=OPENROUTER_MODEL, temperature=SUMMARY_TEMPERATURE,
                     max_tokens=SUMMARY_MAX_TOKENS, api_key=OPENROUTER_API_KEY, workers=WORKERS),
    total=len(chunks) * len(SUMMARY_STYLES), desc="Summarizing",
):
    if "error" in result:
        print(f"[ERROR] {result}")
        errors += 1
    else:
        write_jsonl(str(SUMMARIES_FILE), result)
        written += 1

print(f"Wrote {written} summaries, {errors} errors -> {SUMMARIES_FILE}")

Summarizing 2999 chunks × 3 styles = 8997 API calls


Summarizing:   0%|          | 0/8997 [00:00<?, ?it/s]

Wrote 8997 summaries, 0 errors -> /Users/disipio/development/summarizer-uncertainty-ml/data/summaries_v4.jsonl


## 5. Inspect results

In [5]:
import pandas as pd

records = list(read_jsonl(str(SUMMARIES_FILE)))
df = pd.DataFrame(records)

print(f"Total records : {len(df)}")
print(f"Styles        : {df['summary_style'].value_counts().to_dict()}")
print(f"Pages         : {df['page_title'].nunique()}")
df["src_words"] = df["paragraph_text"].str.split().str.len()
df["tgt_words"] = df["summary"].str.split().str.len()
print("\nSource length (words):")
print(df["src_words"].describe().round(1))
print("\nSummary length (words):")
print(df["tgt_words"].describe().round(1))

Total records : 8997
Styles        : {'shorten': 2999, 'professional': 2999, 'informal': 2999}
Pages         : 77

Source length (words):
count    8997.0
mean      232.8
std        24.1
min        11.0
25%       229.0
50%       238.0
75%       245.0
max       352.0
Name: src_words, dtype: float64

Summary length (words):
count    8997.0
mean       72.2
std        12.0
min        12.0
25%        65.0
50%        72.0
75%        80.0
max       121.0
Name: tgt_words, dtype: float64


In [6]:
# Show one example per style for the first chunk
first_base = df["id"].str.rsplit("|", n=1).str[0].iloc[0]
for _, row in df[df["id"].str.startswith(first_base)].iterrows():
    print(f"=== {row['summary_style'].upper()} ===")
    print(f"SOURCE : {row['paragraph_text'][:200]}...")
    print(f"SUMMARY: {row['summary']}")
    print()

=== SHORTEN ===
SOURCE : A clock or chronometer is a device that measures and displays time. The clock is one of the oldest human inventions, meeting the need to measure intervals of time shorter than the natural units such a...
SUMMARY: A clock is a device that measures and displays time, one of the oldest human inventions designed to measure intervals shorter than natural units like days and months. Early time-measuring instruments include sundials and water clocks, with significant advancements made around 1300 with the invention of mechanical clocks. The pendulum clock, invented by Christiaan Huygens in 1656, marked a major improvement in accuracy.

=== PROFESSIONAL ===
SOURCE : A clock or chronometer is a device that measures and displays time. The clock is one of the oldest human inventions, meeting the need to measure intervals of time shorter than the natural units such a...
SUMMARY: A clock or chronometer is an instrument designed to measure and display time, representing one 